# Notebook 07 — Urban Growth and Built-Up Expansion Analysis
**Stage 8 of the Ghana Groundwater–Subsidence Research Pipeline**

Downloads GHSL built-up surface data (2000–2025), Google Open Buildings (western Ghana tile),
clips to the coastal study area, and computes built-up area, expansion rates, building density,
and urban growth metrics at grid and district level.

In [ ]:
import os
import pathlib; os.chdir(str(pathlib.Path.cwd().parent) if pathlib.Path.cwd().name == 'notebooks' else str(pathlib.Path.cwd()))

import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.transform import rowcol
import rioxarray
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import urllib.request, gzip, io, zipfile, shutil, hashlib
from tqdm import tqdm

ROOT      = Path('.')
GHSL_RAW  = ROOT / 'data' / 'raw' / 'buildings' / 'ghsl'
GOB_RAW   = ROOT / 'data' / 'raw' / 'buildings' / 'gob'
PROC_URB  = ROOT / 'data' / 'processed' / 'urban_growth'
FIG_DIR   = ROOT / 'outputs' / 'figures'
TAB_DIR   = ROOT / 'outputs' / 'tables'
BOUND_DIR = ROOT / 'data' / 'raw' / 'boundaries'
POP_DIR   = ROOT / 'data' / 'interim' / 'harmonized_grids' / 'population'

for d in [GHSL_RAW, GOB_RAW, PROC_URB, FIG_DIR, TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TARGET_CRS = 'EPSG:32630'
STUDY_AREA_FILE = BOUND_DIR / 'study_area_coastal_50km.gpkg'
DISTRICTS_FILE  = BOUND_DIR / 'coastal_districts.gpkg'
GHANA_BBOX_GEO  = (-3.5, 4.0, 1.5, 11.5)  # west, south, east, north

print('Setup OK')
print('Study area file exists:', STUDY_AREA_FILE.exists())


In [ ]:
# 2. Download GHSL GHS_BUILT_S 30-arcsec for key years
GHSL_BASE = 'https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/GHS_BUILT_S_GLOBE_R2023A/'
GHSL_YEARS = [2000, 2010, 2015, 2020, 2025]

def download_file(url, dest, desc=''):
    dest = Path(dest)
    if dest.exists():
        print(f'  Already exists: {dest.name}')
        return dest
    tmp = dest.with_suffix('.tmp')
    req = urllib.request.Request(url)
    req.add_header('User-Agent', 'Mozilla/5.0')
    with urllib.request.urlopen(req, timeout=600) as resp:
        total = int(resp.headers.get('Content-Length', 0))
        done = 0
        with open(tmp, 'wb') as f:
            while True:
                chunk = resp.read(1 << 20)  # 1 MB chunks
                if not chunk:
                    break
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f'  {desc}: {done/1e6:.0f}/{total/1e6:.0f} MB', end='\r')
    tmp.rename(dest)
    print(f'  Downloaded: {dest.name} ({done/1e6:.1f} MB)      ')
    return dest

ghsl_zips = {}
for year in GHSL_YEARS:
    fname = f'GHS_BUILT_S_E{year}_GLOBE_R2023A_4326_30ss_V1_0.zip'
    url = (GHSL_BASE +
           f'GHS_BUILT_S_E{year}_GLOBE_R2023A_4326_30ss/V1-0/{fname}')
    dest = GHSL_RAW / fname
    ghsl_zips[year] = download_file(url, dest, desc=str(year))
    print(f'  {year}: {dest.name}')

print(f'\nAll {len(ghsl_zips)} GHSL ZIPs ready')


In [ ]:
# 3. Unzip, clip to coastal Ghana, reproject to UTM 30N
import re

study_gdf = gpd.read_file(STUDY_AREA_FILE)
study_utm = study_gdf.to_crs(TARGET_CRS)
# Use WGS84 geometry for initial masking
study_wgs = study_gdf.to_crs('EPSG:4326')
shapes_wgs = [geom.__geo_interface__ for geom in study_wgs.geometry]

ghsl_clips = {}  # year -> clipped UTM tif path

for year, zip_path in ghsl_zips.items():
    out_tif = PROC_URB / f'ghsl_builtup_{year}_utm30n.tif'
    if out_tif.exists():
        ghsl_clips[year] = out_tif
        print(f'  {year}: already processed')
        continue

    # Find the TIF inside the ZIP
    with zipfile.ZipFile(zip_path) as zf:
        tif_names = [n for n in zf.namelist() if n.endswith('.tif')]
        if not tif_names:
            print(f'  {year}: no TIF in ZIP')
            continue
        tif_name = tif_names[0]
        # Extract to temp
        tmp_tif = GHSL_RAW / tif_name.split('/')[-1]
        if not tmp_tif.exists():
            zf.extract(tif_name, GHSL_RAW)
            extracted = GHSL_RAW / tif_name
            if extracted != tmp_tif:
                extracted.rename(tmp_tif)

    # Clip to study area in WGS84, then reproject to UTM 30N
    with rasterio.open(tmp_tif) as src:
        out_img, out_transform = rio_mask(src, shapes_wgs, crop=True)
        out_meta = src.meta.copy()
        out_meta.update({'height': out_img.shape[1], 'width': out_img.shape[2],
                         'transform': out_transform, 'crs': src.crs})

    # Reproject to UTM 30N
    with rasterio.MemoryFile() as mem:
        with mem.open(**out_meta) as tmp_ds:
            tmp_ds.write(out_img)
            dst_crs = TARGET_CRS
            transform_utm, width, height = calculate_default_transform(
                tmp_ds.crs, dst_crs, tmp_ds.width, tmp_ds.height, *tmp_ds.bounds)
            meta_utm = tmp_ds.meta.copy()
            meta_utm.update({'crs': dst_crs, 'transform': transform_utm,
                             'width': width, 'height': height})
            with rasterio.open(out_tif, 'w', **meta_utm) as dst:
                reproject(source=rasterio.band(tmp_ds, 1),
                          destination=rasterio.band(dst, 1),
                          src_transform=tmp_ds.transform,
                          src_crs=tmp_ds.crs,
                          dst_transform=transform_utm,
                          dst_crs=dst_crs,
                          resampling=Resampling.bilinear)

    ghsl_clips[year] = out_tif
    size_kb = out_tif.stat().st_size / 1024
    print(f'  {year}: clipped UTM TIF -> {out_tif.name} ({size_kb:.0f} KB)')

print(f'\n{len(ghsl_clips)} GHSL clipped TIFs ready')


In [ ]:
# 4. Download Google Open Buildings tile 0ff (western Ghana, ~31 MB)
GOB_TILE  = '0ff'
GOB_URL   = f'https://storage.googleapis.com/open-buildings-data/v3/polygons_s2_level_4_gzip/{GOB_TILE}_buildings.csv.gz'
gob_gz    = GOB_RAW / f'{GOB_TILE}_buildings.csv.gz'
gob_csv   = GOB_RAW / f'{GOB_TILE}_buildings.csv'

if not gob_gz.exists():
    print(f'Downloading GOB tile {GOB_TILE} ...')
    download_file(GOB_URL, gob_gz, desc=f'GOB_{GOB_TILE}')
else:
    print(f'GOB tile already downloaded: {gob_gz.name} ({gob_gz.stat().st_size/1e6:.1f} MB)')

# Decompress and filter to coastal Ghana bbox
if not gob_csv.exists():
    print('Decompressing...')
    with gzip.open(gob_gz, 'rb') as f_in, open(gob_csv, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print(f'Decompressed: {gob_csv.stat().st_size/1e6:.1f} MB')
else:
    print(f'GOB CSV exists: {gob_csv.stat().st_size/1e6:.1f} MB')

# Load and filter to coastal Ghana
print('Loading GOB CSV...')
gob_df = pd.read_csv(gob_csv, usecols=['latitude','longitude','area_in_meters','confidence'])
print(f'Total buildings in tile: {len(gob_df):,}')
west, south, east, north = GHANA_BBOX_GEO
gob_gh = gob_df[
    (gob_df['latitude'] >= south) & (gob_df['latitude'] <= north) &
    (gob_df['longitude'] >= west) & (gob_df['longitude'] <= east) &
    (gob_df['confidence'] >= 0.65)
].copy()
print(f'Buildings in coastal Ghana (conf>=0.65): {len(gob_gh):,}')
print(f'Area range: {gob_gh["area_in_meters"].min():.0f} – {gob_gh["area_in_meters"].max():,.0f} m²')

# Convert to GeoDataFrame
from shapely.geometry import Point
gob_gdf = gpd.GeoDataFrame(
    gob_gh,
    geometry=gpd.points_from_xy(gob_gh['longitude'], gob_gh['latitude']),
    crs='EPSG:4326'
).to_crs(TARGET_CRS)
gob_gdf.to_file(PROC_URB / 'gob_buildings_westernghana_utm30n.gpkg', driver='GPKG')
print(f'Saved GOB GeoPackage: {len(gob_gdf):,} buildings')


In [ ]:
# 5. Compute built-up area statistics from GHSL time series
PIXEL_AREA_HA = None   # will compute from raster transform

builtup_stats = []
ghsl_arrays   = {}  # year -> 2D numpy array (m² built-up per pixel)

for year in sorted(ghsl_clips.keys()):
    with rasterio.open(ghsl_clips[year]) as src:
        data = src.read(1).astype(float)
        nodata = src.nodata
        if nodata is not None:
            data[data == nodata] = np.nan
        # GHSL GHS_BUILT_S stores built-up area in m² per cell
        # 30 arc-sec at equator ≈ 927m x 927m ≈ 859,329 m² cell area
        # Pixel value = m² built-up surface in that cell
        pixel_res_m2 = abs(src.transform.a * src.transform.e)  # pixel area in map units (m² for UTM)
        if PIXEL_AREA_HA is None:
            PIXEL_AREA_HA = pixel_res_m2 / 10000
            print(f'Pixel area (UTM): {pixel_res_m2/1e4:.2f} ha ({pixel_res_m2:.0f} m²)')
        valid_mask = ~np.isnan(data) & (data >= 0)
        total_built_m2 = np.nansum(data[valid_mask])
        total_built_ha = total_built_m2 / 10000
        n_built_cells  = (data > 0).sum()
        builtup_stats.append({
            'year':             year,
            'built_area_m2':    total_built_m2,
            'built_area_ha':    total_built_ha,
            'built_area_km2':   total_built_ha / 100,
            'n_cells_builtup':  int(n_built_cells),
            'mean_builtup_m2':  float(np.nanmean(data[data > 0])) if n_built_cells > 0 else 0,
        })
        ghsl_arrays[year] = data
        print(f'  {year}: {total_built_ha:,.0f} ha built-up  ({n_built_cells} pixels)')

df_builtup = pd.DataFrame(builtup_stats)

# Built-up growth rate between consecutive periods
df_builtup['growth_ha']   = df_builtup['built_area_ha'].diff()
df_builtup['years_diff']  = df_builtup['year'].diff()
df_builtup['growth_ha_yr']= df_builtup['growth_ha'] / df_builtup['years_diff']
df_builtup['growth_pct']  = (df_builtup['built_area_ha'].pct_change() * 100).round(2)

df_builtup.to_csv(PROC_URB / 'builtup_area_timeseries.csv', index=False)
print('\nBuilt-up area time series:')
print(df_builtup[['year','built_area_km2','growth_ha_yr','growth_pct']].to_string(index=False))


In [ ]:
# 6. Built-up change raster (2000–2025) and 10km grid density

# (a) Change raster: difference between first and last year
yr_first = min(ghsl_arrays.keys())
yr_last  = max(ghsl_arrays.keys())
arr_change = ghsl_arrays[yr_last] - ghsl_arrays[yr_first]

# Save change raster
with rasterio.open(ghsl_clips[yr_last]) as ref:
    meta = ref.meta.copy()
    meta.update({'dtype': 'float32', 'nodata': -9999.0})
    change_path = PROC_URB / f'ghsl_builtup_change_{yr_first}_{yr_last}_utm30n.tif'
    with rasterio.open(change_path, 'w', **meta) as dst:
        out = arr_change.astype('float32')
        out[np.isnan(arr_change)] = -9999.0
        dst.write(out, 1)
print(f'Change raster saved: {change_path.name}')

# (b) Aggregate GHSL 2020 and 2000 to 10km analysis grid
grid_10km = gpd.read_file(BOUND_DIR / 'analysis_grid_10km.gpkg').to_crs(TARGET_CRS)
print(f'10km grid cells: {len(grid_10km)}')

def raster_sum_by_polygon(raster_path, polygons_gdf):
    """Sum raster values within each polygon."""
    sums = []
    with rasterio.open(raster_path) as src:
        for geom in polygons_gdf.geometry:
            try:
                out_img, _ = rio_mask(src, [geom.__geo_interface__], crop=True)
                vals = out_img[0].astype(float)
                nd = src.nodata
                if nd is not None:
                    vals[vals == nd] = np.nan
                sums.append(np.nansum(vals))
            except Exception:
                sums.append(np.nan)
    return sums

print('Aggregating GHSL to 10km grid (2000 and 2020)...')
grid_10km['builtup_m2_2000'] = raster_sum_by_polygon(ghsl_clips[2000], grid_10km)
grid_10km['builtup_m2_2020'] = raster_sum_by_polygon(ghsl_clips[2020], grid_10km)
grid_10km['builtup_m2_2025'] = raster_sum_by_polygon(ghsl_clips[2025], grid_10km)
grid_10km['builtup_change_m2'] = grid_10km['builtup_m2_2025'] - grid_10km['builtup_m2_2000']
grid_10km['builtup_growth_pct'] = (
    (grid_10km['builtup_m2_2025'] - grid_10km['builtup_m2_2000'])
    / (grid_10km['builtup_m2_2000'] + 1) * 100
).round(2)
# Built-up density 2020 (fraction of cell that is built up)
cell_area_m2 = (10000 * 10000)  # 10km x 10km = 1e8 m²
grid_10km['builtup_density_2020'] = (grid_10km['builtup_m2_2020'] / cell_area_m2).clip(0, 1)

grid_10km.to_file(PROC_URB / 'builtup_grid10km.gpkg', driver='GPKG')
print(f'Saved 10km grid with built-up stats: {len(grid_10km)} cells')


In [ ]:
# 7. Population density by 10km grid (WorldPop 2020)
pop_2020_file = POP_DIR / 'gha_ppp_2020.tif'
if pop_2020_file.exists():
    grid_10km['population_2020'] = raster_sum_by_polygon(pop_2020_file, grid_10km)
    total_pop = grid_10km['population_2020'].sum()
    print(f'Total coastal Ghana population (2020): {total_pop:,.0f}')
    grid_10km['pop_density_km2'] = grid_10km['population_2020'] / 100  # 10km x 10km = 100 km²
    grid_10km.to_file(PROC_URB / 'builtup_grid10km.gpkg', driver='GPKG')
    print('Population column added to grid')
else:
    print(f'WorldPop 2020 not found at {pop_2020_file}')
    grid_10km['population_2020'] = np.nan
    grid_10km['pop_density_km2'] = np.nan


In [ ]:
# 8. Urban expansion rate table and hotspot identification
# Label high-growth cells (top quartile built-up growth %)
q75 = grid_10km['builtup_growth_pct'].quantile(0.75)
grid_10km['urban_hotspot'] = (grid_10km['builtup_growth_pct'] >= q75).astype(int)

# Summary table
cols = ['builtup_m2_2000','builtup_m2_2020','builtup_m2_2025',
        'builtup_growth_pct','builtup_density_2020','population_2020','pop_density_km2']
summary = grid_10km[cols].describe().round(2)
print(summary.to_string())

# Urban growth rate CSV
df_builtup.to_csv(PROC_URB / 'urban_expansion_rate.csv', index=False)

# Building density from GOB (western Ghana only)
print(f'\nGOB buildings in coastal Ghana: {len(gob_gdf):,}')
print(f'Mean building area: {gob_gdf["area_in_meters"].mean():.1f} m²')
print(f'Median building area: {gob_gdf["area_in_meters"].median():.1f} m²')


In [ ]:
# 9. Figure: built-up area time series
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('GHSL Built-Up Surface — Coastal Ghana 2000–2025', fontsize=12, fontweight='bold')

ax = axes[0]
ax.bar(df_builtup['year'], df_builtup['built_area_km2'],
       color='#d6604d', alpha=0.85, width=3)
ax.set_xlabel('Year'); ax.set_ylabel('Built-up area (km²)')
ax.set_title('Total Built-Up Area', fontsize=10)
ax.grid(axis='y', alpha=0.3)
for _, row in df_builtup.iterrows():
    ax.text(row['year'], row['built_area_km2'] + 5, f"{row['built_area_km2']:.0f}",
            ha='center', va='bottom', fontsize=8)

ax = axes[1]
growths = df_builtup[df_builtup['growth_ha_yr'].notna()]
colours = ['#d6604d' if v > 0 else '#4393c3' for v in growths['growth_ha_yr']]
ax.bar(growths['year'], growths['growth_ha_yr'], color=colours, alpha=0.85, width=3)
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel('Year'); ax.set_ylabel('Built-up growth (ha/year)')
ax.set_title('Annual Built-Up Growth Rate', fontsize=10)
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'builtup_timeseries.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'builtup_timeseries.png')


In [ ]:
# 10. Figure: urban growth map (built-up change 2000-2025)
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import TwoSlopeNorm

fig, axes = plt.subplots(1, 2, figsize=(15, 8),
                          subplot_kw={'projection': ccrs.PlateCarree()})
fig.suptitle('Urban Growth — Coastal Ghana (GHSL Built-Up Surface)',
             fontsize=12, fontweight='bold')

# Panel A: built-up density 2020
ax = axes[0]
grid_4326 = grid_10km.to_crs('EPSG:4326')
grid_4326['centroid_x'] = grid_4326.geometry.centroid.x
grid_4326['centroid_y'] = grid_4326.geometry.centroid.y

sc1 = ax.scatter(grid_4326['centroid_x'], grid_4326['centroid_y'],
                 c=grid_4326['builtup_density_2020'], cmap='YlOrRd',
                 s=60, vmin=0, vmax=0.3, alpha=0.85,
                 transform=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, lw=0.7)
ax.add_feature(cfeature.BORDERS, lw=0.4, linestyle=':')
ax.add_feature(cfeature.LAND, facecolor='#f0f0f0', zorder=0)
ax.set_extent([-3.5, 1.5, 4.0, 11.5], crs=ccrs.PlateCarree())
cbar1 = fig.colorbar(sc1, ax=ax, shrink=0.65)
cbar1.set_label('Built-up fraction 2020', fontsize=9)
ax.set_title('A  Built-Up Density 2020', loc='left', fontsize=10)
gl = ax.gridlines(draw_labels=True, lw=0.3, alpha=0.5)
gl.top_labels = gl.right_labels = False

# Panel B: built-up growth % (2000-2025)
ax = axes[1]
vmax_g = grid_4326['builtup_growth_pct'].quantile(0.95)
sc2 = ax.scatter(grid_4326['centroid_x'], grid_4326['centroid_y'],
                 c=grid_4326['builtup_growth_pct'].clip(0, vmax_g),
                 cmap='Reds', s=60, vmin=0, vmax=vmax_g, alpha=0.85,
                 transform=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, lw=0.7)
ax.add_feature(cfeature.BORDERS, lw=0.4, linestyle=':')
ax.add_feature(cfeature.LAND, facecolor='#f0f0f0', zorder=0)
ax.set_extent([-3.5, 1.5, 4.0, 11.5], crs=ccrs.PlateCarree())
cbar2 = fig.colorbar(sc2, ax=ax, shrink=0.65)
cbar2.set_label('Built-up growth 2000–2025 (%)', fontsize=9)
ax.set_title('B  Built-Up Growth Rate 2000–2025', loc='left', fontsize=10)
gl2 = ax.gridlines(draw_labels=True, lw=0.3, alpha=0.5)
gl2.top_labels = gl2.right_labels = False

fig.tight_layout()
fig.savefig(FIG_DIR / 'urban_growth_map.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'urban_growth_map.png')


In [ ]:
# 11. Stage 8 summary
print('=' * 60)
print('Stage 8 complete')
outputs = [
    PROC_URB / 'ghsl_builtup_2000_utm30n.tif',
    PROC_URB / 'ghsl_builtup_2020_utm30n.tif',
    PROC_URB / 'ghsl_builtup_2025_utm30n.tif',
    PROC_URB / f'ghsl_builtup_change_{yr_first}_{yr_last}_utm30n.tif',
    PROC_URB / 'gob_buildings_westernghana_utm30n.gpkg',
    PROC_URB / 'builtup_grid10km.gpkg',
    PROC_URB / 'builtup_area_timeseries.csv',
    PROC_URB / 'urban_expansion_rate.csv',
    FIG_DIR  / 'builtup_timeseries.png',
    FIG_DIR  / 'urban_growth_map.png',
]
for p in outputs:
    tick = 'OK' if Path(p).exists() else 'MISSING'
    print(f'  [{tick}]  {Path(p).name}')

total_built_2020 = df_builtup.loc[df_builtup['year']==2020,'built_area_km2'].values[0]
total_built_2025 = df_builtup.loc[df_builtup['year']==2025,'built_area_km2'].values[0]
print()
print(f'Built-up area 2000: {df_builtup.loc[df_builtup["year"]==2000,"built_area_km2"].values[0]:.1f} km²')
print(f'Built-up area 2020: {total_built_2020:.1f} km²')
print(f'Built-up area 2025: {total_built_2025:.1f} km²')
print(f'GOB buildings (western Ghana): {len(gob_gdf):,}')
